### Create AnnData object for GSE73173 — Yui 2018 colonic regeneration microarray

- **Developed by:** Anna Maguza
- **Affiliation:** Faculty of Medicine, Würzburg University
- **Date of creation:** 7 May 2026
- **Last modified date:** 7 May 2026

Builds an AnnData (samples × genes) from 12 Agilent **single-channel** microarray FE files at `LGR5_analysis_data/GSE73173/GSE73173_RAW/` (platform GPL13912 — Agilent SurePrint G3 Mouse GE 8×60K). Yui 2018 design: bulk colonic epithelium during DSS-induced colitis regeneration; Lgr5+ ISC genes are followed transcriptionally rather than by FACS-sorting.

Because the cells are **not FACS-gated** on Lgr5, every sample gets `lgr5_status='unknown'` — Lgr5 lives in `var`, not `obs`. The DSS-injury time course goes into `condition`, but the per-GSM timepoint assignment is **best-guess by GSM order**; verify against GEO (`https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE73173`) before relying on this AnnData. The science doc claims this is bulk RNA-seq — that is incorrect; the deposited files are Agilent microarray (see `LGR5_data_folder_inventory.md`).

### Import packages

In [1]:
import os, sys, glob
from datetime import datetime

import anndata as ad
import pandas as pd
import scanpy as sc

sys.path.insert(0, os.getcwd())
from utils.agilent_fe import load_agilent_study

/Users/am336941/uv_envs/pyenv313/.venv/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/Users/am336941/uv_envs/pyenv313/.venv/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/Users/am336941/uv_envs/pyenv313/.venv/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


### Per-sample metadata

12 samples deposited as `GSM1888139`–`GSM1888150`. Per-GSM timepoint assignment is unknown without GEO sample metadata — set as `unknown` placeholder in `condition`; the user should overwrite from the GEO record.

In [2]:
DATA_DIR = '/Users/am336941/PhD/data/LGR5_analysis_data/GSE73173/GSE73173_RAW'

GSMS = [f'GSM{i}' for i in range(1888139, 1888151)]
SAMPLE_PATHS = {}
for gsm in GSMS:
    matches = glob.glob(os.path.join(DATA_DIR, f'{gsm}_*.txt.gz'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {gsm}_*.txt.gz in {DATA_DIR}, got {matches}')
    SAMPLE_PATHS[gsm] = matches[0]

SAMPLE_META = {
    gsm: dict(
        sample=gsm,
        lgr5_status='unknown',
        lgr5_label_raw='unsorted-bulk',
        condition='unknown',  # ← TODO: overwrite with timepoint (uninjured / DSS-day-N / regenerated) from GEO
        cell_type='colonic epithelium',
        tissue='colon',
        GSE='GSE73173',
        organism='mus musculus',
        technology='Agilent 1-color microarray (GPL13912)',
        assay_modality='microarray',
    )
    for gsm in GSMS
}

### Parse and build AnnData

1-channel array → use `gProcessedSignal` rather than `LogRatio`.

In [3]:
adata = load_agilent_study(SAMPLE_PATHS, SAMPLE_META, value_column='gProcessedSignal', aggregate='mean')
for col in ['sample', 'lgr5_status', 'lgr5_label_raw', 'condition', 'cell_type', 'tissue', 'GSE', 'organism', 'technology', 'assay_modality']:
    adata.obs[col] = adata.obs[col].astype('category')
adata

AnnData object with n_obs × n_vars = 12 × 34797
    obs: 'sample', 'lgr5_status', 'lgr5_label_raw', 'condition', 'cell_type', 'tissue', 'GSE', 'organism', 'technology', 'assay_modality'
    var: 'n_probes'
    uns: 'agilent_value_column', 'agilent_aggregate', 'agilent_n_channels'

### Save

In [4]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
adata.uns['GSE'] = 'GSE73173'
adata.uns['publication'] = 'Yui S et al., Cell Stem Cell 22:35-49 (2018) — YAP/TAZ-dependent reprogramming of colonic epithelium'
adata.uns['genome_reference'] = 'GPL13912 — Agilent SurePrint G3 Mouse GE 8x60K (probe annotation in agilent_fe parser output)'
adata.uns['source_files'] = sorted([os.path.basename(p) for p in SAMPLE_PATHS.values()] + ['GPL13912_old_annotations.txt.gz'])
adata.uns['processing_history'] = {
    timestamp: 'AnnData created from 12 Agilent 1-channel FE files; gProcessedSignal extracted; per-gene mean across probes; lgr5_status=unknown (Lgr5 used as a transcriptional readout, not a FACS gate). | note: X holds processed signal intensities, NOT log-ratios. Per-GSM DSS-injury timepoint not yet annotated — see TODO in obs.condition.',
}

out_dir = 'data/LGR5_analysis'
os.makedirs(out_dir, exist_ok=True)
out_path = f'{out_dir}/gut_mm_GSE73173_AM_{timestamp}_raw.h5ad'
adata.write_h5ad(out_path)
print(out_path, adata.shape)

data/LGR5_analysis/gut_mm_GSE73173_AM_07052026_233404_raw.h5ad (12, 34797)
